# 00 - Setup & Connection Tests

Validates that your local environment can reach **S3**, **Athena**, and **RDS** on AWS.

## Prerequisites

1. Copy `.env.example` to `.env` and fill in your credentials
2. Install dependencies: `pip install -r requirements-dev.txt`
3. Authenticate AWS SSO: `aws sso login --profile <your-profile>`

In [ ]:
# Install deps (run once)
# !pip install -r requirements-dev.txt

In [ ]:
import sys, os

# Add notebooks dir to path for local_helpers
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from local_helpers import (
    test_connections,
    get_aws_session,
    s3_client,
    list_s3_prefix,
    read_s3_json,
    athena_query,
    rds_query,
    get_rds_engine,
)

print('Imports OK')

## 1. Run all connection tests

In [ ]:
test_connections()

## 2. S3 - Browse Iceberg warehouse

In [ ]:
bucket = os.getenv('RS_ENGINE_BUCKET', 'finomics-data-pod')

# List top-level warehouse contents
keys = list_s3_prefix(bucket, 'warehouse/', max_keys=20)
for k in keys:
    print(k)

In [ ]:
# Load a config JSON from S3
provider_config = read_s3_json(bucket, 'warehouse/scripts/common/apatel/provider_config.json')
print(f'Provider config top-level keys: {list(provider_config.keys())}')

## 3. Athena - Query Iceberg tables

In [ ]:
# List all tables in the Iceberg database
db = os.getenv('ICEBERG_DB', 'finomics_catalog_data')
tables_df = athena_query(f"SHOW TABLES IN {db}")
tables_df

In [ ]:
# Quick count of SKU catalog entries
athena_query("""
    SELECT cloud_provider, COUNT(*) as cnt
    FROM rs_cloud_sku_catalog
    WHERE is_active = TRUE
    GROUP BY cloud_provider
""")

## 4. RDS - Query PostgreSQL

In [ ]:
# List tables in public schema
rds_query("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public' 
    ORDER BY table_name
""")

In [ ]:
# Sample custom_recommendations
rds_query("""
    SELECT cloud_provider, rec_type, COUNT(*) as cnt
    FROM custom_recommendations
    GROUP BY cloud_provider, rec_type
    ORDER BY cnt DESC
    LIMIT 20
""")

## 5. AWS Identity Check

In [ ]:
sts = get_aws_session().client('sts')
identity = sts.get_caller_identity()
print(f"Account: {identity['Account']}")
print(f"ARN:     {identity['Arn']}")